In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Feedforward clássico e fluxo de controle (circuitos dinâmicos)

<Accordion>
<AccordionItem title="Package versions">

O código nesta página foi desenvolvido usando os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
Os circuitos dinâmicos são ferramentas poderosas com as quais você pode medir qubits no meio da execução de um circuito quântico e, em seguida, realizar operações de lógica clássica dentro do circuito, com base nos resultados dessas medições intermediárias. Esse processo também é conhecido como _feedforward clássico_. Embora ainda seja cedo para entender como melhor aproveitar os circuitos dinâmicos, a comunidade de pesquisa quântica já identificou vários casos de uso, como os seguintes:

* Preparação eficiente de estados quânticos, como o [estado GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), [estado W](https://arxiv.org/abs/2403.07604) (para mais informações sobre o estado W, consulte também ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), e uma ampla classe de [estados de produto matricial](https://arxiv.org/abs/2404.16083)
* [Emaranhamento eficiente de longo alcance](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) entre qubits no mesmo chip usando circuitos rasos
* [Amostragem eficiente de circuitos similares a IQP](https://arxiv.org/pdf/2505.04705)
## Instrução `if`
A instrução `if` é usada para realizar operações condicionalmente com base no valor de um bit ou registrador clássico.

No exemplo abaixo, aplicamos uma gate Hadamard a um qubit e o medimos. Se o resultado for 1, aplicamos uma gate X no qubit, que tem o efeito de revertê-lo para o estado 0. Em seguida, medimos o qubit novamente. O resultado da medição deve ser 0 com 100% de probabilidade.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/60924bfa-50ed-4d9d-a17b-9d64f2cc053f-0.svg)

A instrução `with` pode receber um alvo de atribuição que é, por si só, um gerenciador de contexto que pode ser armazenado e posteriormente usado para criar um bloco else, que é executado sempre que o conteúdo do bloco `if` *não* é executado.

No exemplo abaixo, inicializamos registradores com dois qubits e dois bits clássicos. Aplicamos uma gate Hadamard ao primeiro qubit e o medimos. Se o resultado for 1, aplicamos uma gate Hadamard no segundo qubit; caso contrário, aplicamos uma gate X no segundo qubit. Por fim, medimos o segundo qubit também.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/20f0640a-a3f7-41b3-aada-b66bc89b0555-0.svg)

Além de condicionar em um único bit clássico, também é possível condicionar no valor de um registrador clássico composto por múltiplos bits.

No exemplo abaixo, aplicamos gates Hadamard a dois qubits e os medimos. Se o resultado for `01`, ou seja, o primeiro qubit é 1 e o segundo qubit é 0, então aplicamos uma gate X a um terceiro qubit. Por fim, medimos o terceiro qubit. Observe que, para maior clareza, optamos por especificar o estado do terceiro bit clássico, que é 0, na condição `if`. No desenho do circuito, a condição é indicada pelos círculos nos bits clássicos sobre os quais a condição é aplicada. Um círculo sólido indica condicionamento em 1, enquanto um círculo sem preenchimento indica condicionamento em 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/98e8f552-4169-42a3-8182-e14e9ffb59e2-0.svg)

## Expressões clássicas
O módulo de expressões clássicas do Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) contém uma representação exploratória de operações em tempo de execução sobre valores clássicos durante a execução do circuito. Devido a limitações de hardware, apenas as condições `QuantumCircuit.if_test()` são atualmente suportadas.

O exemplo a seguir mostra que você pode usar o cálculo de paridade para criar um estado GHZ de n qubits usando circuitos dinâmicos. Primeiro, gere $n/2$ pares de Bell em qubits adjacentes. Em seguida, una esses pares usando uma camada de gates CNOT entre os pares. Você então mede o qubit alvo de todas as gates CNOT anteriores e reinicia cada qubit medido para o estado $\vert 0 \rangle$. Você aplica $X$ a cada site não medido para o qual a paridade de todos os bits precedentes é ímpar. Por fim, as gates CNOT são aplicadas aos qubits medidos para restabelecer o emaranhamento perdido na medição.

No cálculo de paridade, o primeiro elemento da expressão construída envolve elevar o objeto Python `mr[0]` a um nó [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (`lift` é usado para transformar objetos arbitrários em expressões clássicas). Isso não é necessário para `mr[1]` e o possível registrador clássico seguinte, pois eles são entradas para `expr.bit_xor`, e qualquer elevação necessária é feita automaticamente nesses casos. Tais expressões podem ser construídas em loops e outras estruturas.

In [4]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [5]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg" alt="Output of the previous code cell" />

![Saída da célula de código anterior](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d0f0abdb-50d5-408d-a704-a1a555acdd85-0.svg)

<span id="store"></span>
### `Store`

Você pode usar a instrução [`store`](https://docs.quantum.ibm.com/api/qiskit/circuit#store) para salvar o resultado de uma expressão clássica, caso essa expressão seja usada repetidamente. As operações são paralelizadas automaticamente, tornando seu código significativamente mais eficiente em tempo de execução.

Por exemplo, é mais natural e mais eficiente em tempo de execução escrever $B[0] \oplus B[1] \oplus B[2] \ldots$, onde $B = \neg A$, do que $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$. O primeiro computa a negação em um único passo paralelo antes da cadeia XOR, em vez de avaliar cada negação sequencialmente dentro da expressão.

Exemplo completo:

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

qregs = QuantumRegister(4, "q")
creg = ClassicalRegister(3, "c")
# temp is a plain ClassicalRegister used as the store target
temp = ClassicalRegister(3, "temp")
qc = QuantumCircuit(qregs, creg, temp)

qc.h([0, 1, 2])
qc.measure([0, 1, 2], creg)

# Store bit-NOT of the full 3-bit register into temp
qc.store(temp, expr.bit_not(creg))

# Compute parity of temp using bit-indexed XOR
parity = expr.bit_xor(
    expr.bit_xor(expr.index(temp, 0), expr.index(temp, 1)),
    expr.index(temp, 2),
)

# Flip q3 if parity of ~creg is 1
with qc.if_test(parity):
    qc.x(3)

qc.measure([0, 1, 2], creg)

qc.draw("mpl")

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/e64ec241-41e8-40f8-ab64-af236c6c7802-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/e64ec241-41e8-40f8-ab64-af236c6c7802-0.avif)

## Próximas etapas
> **Tip:** - Aprenda a implementar desacoplamento dinâmico preciso usando [stretch](/guides/stretch).
> - Use a [visualização do cronograma do circuito](/guides/qiskit-runtime-circuit-timing) para depurar e otimizar seus circuitos dinâmicos.
> - [Execute circuitos dinâmicos](/guides/execute-dynamic-circuits).